# Figuras del Capítulo 4 — Compresión SVD

Genera todas las figuras (versiones `es` y `en`) del capítulo 4 a partir de los
CSVs exportados por los notebooks 02 (`results/notebook2/`) y 03
(`results/notebook3/`).

**Figuras producidas (en `figures/`):**
1. `cap4_anatomy_heatmap_{es,en}.png` — k95 por capa × componente
2. `cap4_spectral_fingerprint_{es,en}.png` — decaimiento espectral por tipo
3. `cap4_uniform_transition_{es,en}.png` — transición de fase uniforme
4. `cap4_emotion_vulnerability_{es,en}.png` — F1 por emoción × rango
5. `cap4_component_sensitivity_{es,en}.png` — sensibilidad componente a r=128
6. `cap4_depth_sensitivity_{es,en}.png` — sensibilidad por profundidad
7. `cap4_pareto_frontier_{es,en}.png` — frontera Pareto de estrategias
8. `cap4_adaptive_rank_distribution_{es,en}.png` — rangos adaptativos
9. `cap4_qkv_overlap_{es,en}.png` — solapamiento de subespacios QKV

Puede ejecutarse tanto localmente (desde `plots/`) como en Colab.

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    PLOTS_DIR = os.path.join(PROJECT_ROOT, 'plots')
else:
    # Local: notebook lives in plots/, project root is one level up
    PLOTS_DIR = os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != 'plots' else os.getcwd())
    PROJECT_ROOT = os.path.abspath(os.path.join(PLOTS_DIR, '..')) if os.path.basename(PLOTS_DIR) == 'plots' else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if not os.path.basename(PLOTS_DIR) == 'plots':
        PLOTS_DIR = os.path.join(PROJECT_ROOT, 'plots')
    os.chdir(PLOTS_DIR)
    if PLOTS_DIR not in sys.path:
        sys.path.insert(0, PLOTS_DIR)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'PLOTS_DIR:    {PLOTS_DIR}')
print(f'CWD:          {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tfg_plot_style as sty
sty.apply(lang='es')

# CSV paths (refactored NB02/NB03 outputs)
NB02_DIR = os.path.join(PROJECT_ROOT, 'results', 'notebook2')
NB03_DIR = os.path.join(PROJECT_ROOT, 'results', 'notebook3')

# NB02 outputs
CSV_K95_MATRIX       = os.path.join(NB02_DIR, 'rank_matrix_k95.csv')
CSV_SINGULAR_VALUES  = os.path.join(NB02_DIR, 'singular_values_by_component.csv')
CSV_UNIFORM_RESULTS  = os.path.join(NB02_DIR, 'uniform_compression_results.csv')
CSV_UNIFORM_EMOTIONS = os.path.join(NB02_DIR, 'uniform_emotion_f1.csv')
CSV_ADAPTIVE_RANKS   = os.path.join(NB02_DIR, 'adaptive_rank_distribution.csv')
CSV_MASTER_STRATEGY  = os.path.join(NB02_DIR, 'spectral_analysis_results.csv')
CSV_QKV_OVERLAP      = os.path.join(NB02_DIR, 'qkv_subspace_overlap.csv')

# NB03 outputs
CSV_COMP_SENSITIVITY  = os.path.join(NB03_DIR, 'component_sensitivity.csv')
CSV_DEPTH_SENSITIVITY = os.path.join(NB03_DIR, 'depth_sensitivity.csv')

os.makedirs('figures', exist_ok=True)


def load_csv(path, fallback=None, name=''):
    try:
        df = pd.read_csv(path)
        print(f'[ok] {name}: {path} -- {df.shape}')
        return df
    except FileNotFoundError:
        if fallback is not None:
            print(f'[!]  {name}: not found -- using fallback')
            return fallback
        print(f'[X]  {name}: {path} -- NOT FOUND')
        return None

## Figura 1 — Anatomía del modelo (k95 por capa × componente)

In [ ]:
k95_df = load_csv(CSV_K95_MATRIX, name='k95 matrix')

if k95_df is not None:
    def plot_anatomy(ax, df=k95_df):
        df = df.set_index(df.columns[0])
        data = df.values.astype(float)
        comp_names = list(df.columns)
        n_layers = data.shape[0]

        plot_df = pd.DataFrame(data, index=[f'L{i}' for i in range(n_layers)], columns=comp_names)
        sns.heatmap(plot_df, annot=True, fmt='.0f', cmap=sty.CMAP_COOL,
                    vmin=150, vmax=700,
                    linewidths=0.8, linecolor=sty.BG,
                    cbar_kws={'label': '$k_{95}$', 'shrink': 0.8},
                    annot_kws={'size': sty.SMALL_SIZE},
                    ax=ax)

        for text in ax.texts:
            val = int(text.get_text())
            text.set_color('white' if val > 500 else sty.INK)

        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=sty.TICK_SIZE)

        ax.set_ylabel(sty.L['layer'])
        ax.set_xlabel(sty.L['component'])
        ax.set_title(sty.L['t_anatomy'])
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=sty.TICK_SIZE)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=sty.TICK_SIZE)

        ffn_idx = next((i for i, c in enumerate(comp_names) if 'FFN' in c or 'ffn' in c), 4)
        ax.axvline(x=ffn_idx, color=sty.INK, lw=1.5, ls='--', alpha=0.4)
        ax.text(ffn_idx / 2, -0.6, sty.L['self_attention'], ha='center',
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style='italic')
        ax.text((ffn_idx + len(comp_names)) / 2, -0.6, sty.L['feed_forward'], ha='center',
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style='italic')

    sty.generate_both(plot_anatomy, 'anatomy_heatmap', figsize=sty.FIG_FULL)
else:
    print('-> Run NB02 export to produce rank_matrix_k95.csv')

## Figura 2 — Huella espectral (decaimiento singular por tipo)

In [ ]:
sv_df = load_csv(CSV_SINGULAR_VALUES, name='singular values')

if sv_df is not None:
    def plot_spectral(ax, df=sv_df):
        mean_cols = [c for c in df.columns if c.endswith('_mean')]
        x = df['rank_index'].values if 'rank_index' in df.columns else np.arange(1, len(df)+1)

        for col in mean_cols:
            comp_name = col.replace('_mean', '').replace('_', ' ')
            std_col = col.replace('_mean', '_std')
            color = sty.component_color(comp_name)
            lw = 2.0 if 'FFN' in comp_name else 1.5
            ls = '-' if any(k in comp_name for k in ('Query', 'Key', 'FFN')) else '--'

            ax.plot(x, df[col], color=color, lw=lw, ls=ls, label=comp_name, alpha=0.85)
            if std_col in df.columns:
                ax.fill_between(x, df[col] - df[std_col], df[col] + df[std_col],
                                color=color, alpha=0.08)

        ax.set_xlabel(sty.L['singular_index'])
        ax.set_ylabel(sty.L['singular_magnitude'])
        ax.set_title(sty.L['t_spectral'])
        ax.legend(loc='upper right', ncol=2)
        ax.set_xlim(0, None)
        ax.set_ylim(-0.02, 1.05)

    sty.generate_both(plot_spectral, 'spectral_fingerprint', figsize=sty.FIG_FULL)
else:
    print('-> Run NB02 export to produce singular_values_by_component.csv')

## Figura 3 — Transición de fase bajo compresión uniforme

In [ ]:
# NB02 writes column names: strategy, rank, macro_f1, retention, compression_ratio
uni_fallback = pd.DataFrame({
    'strategy':  ['Baseline', 'Uniform r=512', 'Uniform r=384', 'Uniform r=256',
                  'Uniform r=128', 'Uniform r=64'],
    'rank':      [768,   512,   384,   256,   128,   64],
    'macro_f1':  [0.577, 0.464, 0.251, 0.025, 0.000, 0.000],
    'retention': [1.000, 0.805, 0.435, 0.043, 0.000, 0.000],
})
uni_df = load_csv(CSV_UNIFORM_RESULTS, fallback=uni_fallback, name='uniform results')


def plot_transition(ax, df=uni_df):
    ranks = df['rank'].values
    retention = df['retention'].values

    ax.plot(ranks, retention, color=sty.BLUE, marker='s', markersize=8, lw=2.2,
            markeredgecolor='white', markeredgewidth=1.2, zorder=5)

    ax.axvspan(0, 256, alpha=0.04, color=sty.TERRA, zorder=0)
    ax.text(160, 0.88, sty.L['collapse_zone'], ha='center',
            fontsize=sty.ANNOTATION_SIZE, color=sty.TERRA_L, style='italic')
    ax.axvspan(256, 384, alpha=0.04, color=sty.SAND, zorder=0)
    ax.annotate(sty.L['phase_transition'], xy=(320, 0.24),
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, ha='center', style='italic')

    for r, ret in zip(ranks, retention):
        if 64 <= r <= 512 and ret > 0.01:
            offset = (10, 10) if ret > 0.1 else (10, -15)
            sty.annotate_point(ax, r, ret, f'{ret:.1%}', offset=offset)

    ax.set_xlabel(sty.L['rank'])
    ax.set_ylabel(sty.L['f1_retention'])
    ax.set_title(sty.L['t_uniform'])
    ax.set_xlim(0, 800)
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)
    ax.invert_xaxis()


sty.generate_both(plot_transition, 'uniform_transition', figsize=sty.FIG_WIDE)

## Figura 4 — F1 por emoción × nivel de compresión uniforme

In [ ]:
# Expected columns: emotion, Baseline, Uniform r=512, ..., Uniform r=64
emo_df = load_csv(CSV_UNIFORM_EMOTIONS, name='uniform emotion F1')

if emo_df is not None:
    def plot_emotion_vuln(ax, df=emo_df):
        if df.columns[0].lower() in ('emotion', 'emocion'):
            df = df.set_index(df.columns[0])

        # Keep only uniform / baseline columns
        uniform_cols = [c for c in df.columns
                        if 'baseline' in c.lower() or 'uniform' in c.lower()
                        or c.startswith('r=') or c.startswith('r ')]
        if uniform_cols:
            df = df[uniform_cols]

        df = df.sort_values(df.columns[0], ascending=True)

        sns.heatmap(df, annot=True, fmt='.2f', cmap='YlOrRd',
                    vmin=0, vmax=1,
                    linewidths=0.8, linecolor=sty.BG,
                    cbar_kws={'label': 'F1', 'shrink': 0.6},
                    annot_kws={'size': 6.5},
                    ax=ax)

        for text in ax.texts:
            if text.get_text() == '0.00':
                text.set_text('')

        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=sty.TICK_SIZE)
        cbar.set_label('F1', fontsize=sty.LABEL_SIZE)

        ax.set_title(sty.L['t_emotion_vuln'])
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=sty.TICK_SIZE)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

    sty.generate_both(plot_emotion_vuln, 'emotion_vulnerability', figsize=(5.5, 7.5))
else:
    print('-> Run NB02 export to produce uniform_emotion_f1.csv')

## Figura 5 — Sensibilidad por componente a rango r=128

NB03 `component_sensitivity.csv` columns: `component, rank, f1_macro,
f1_retention, f1_retention_pct, silhouette` — `f1_retention` is already a
fraction in [0, 1].

In [ ]:
comp_raw = load_csv(CSV_COMP_SENSITIVITY, name='NB03 component sensitivity')

if comp_raw is not None:
    comp_128 = comp_raw[comp_raw['rank'] == 128].copy()
    # f1_retention is already a fraction in the refactored NB03
    comp_128['retention'] = comp_128['f1_retention']
    comp_128 = comp_128.sort_values('retention', ascending=False)
    print(f'Components at r=128: {len(comp_128)}')
    print(comp_128[['component', 'f1_macro', 'retention']].to_string(index=False))
else:
    comp_128 = pd.DataFrame({
        'component': ['Query (Q)', 'Key (K)', 'Value (V)', 'Attn Output',
                      'FFN Output', 'FFN Inter'],
        'f1_macro':  [0.574, 0.568, 0.395, 0.325, 0.128, 0.040],
        'retention': [0.994, 0.984, 0.685, 0.563, 0.222, 0.069],
    })

In [ ]:
def plot_comp_sens(ax, df=comp_128):
    comps = df['component'].values
    retentions = df['retention'].values
    colors = [sty.component_color(c) for c in comps]

    ax.barh(range(len(comps)), retentions, color=colors, height=0.55,
            edgecolor=sty.BG, linewidth=0.8)

    for i, (comp, ret) in enumerate(zip(comps, retentions)):
        x_pos = ret + 0.015 if ret < 0.85 else ret - 0.07
        color = sty.INK_2 if ret < 0.85 else 'white'
        ax.text(x_pos, i, f'{ret:.1%}', va='center', fontsize=sty.ANNOTATION_SIZE,
                color=color, weight='medium')

    ax.set_yticks(range(len(comps)))
    ax.set_yticklabels(comps)
    ax.set_xlabel(sty.L['f1_retention'])
    ax.set_title(sty.L['t_comp_sens'])
    ax.set_xlim(0, 1.12)
    sty.format_pct(ax, axis='x')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    ax.grid(axis='y', visible=False)
    ax.axvline(x=0.80, color=sty.INK_3, ls=':', lw=0.7, alpha=0.5)


sty.generate_both(plot_comp_sens, 'component_sensitivity', figsize=sty.FIG_WIDE)

## Figura 6 — Sensibilidad por grupo de profundidad

NB03 `depth_sensitivity.csv` columns: `depth_group, rank, f1_macro,
f1_retention, f1_retention_pct, silhouette`.

In [ ]:
depth_raw = load_csv(CSV_DEPTH_SENSITIVITY, name='NB03 depth sensitivity')

if depth_raw is not None:
    depth_raw = depth_raw.copy()
    # f1_retention is already a fraction
    depth_raw['retention'] = depth_raw['f1_retention']
    depth_pivot = depth_raw.pivot(index='depth_group', columns='rank', values='retention')
    depth_pivot.columns = [f'r{int(c)}' for c in depth_pivot.columns]
    depth_pivot = depth_pivot.reset_index()
    depth_pivot.columns = ['group'] + list(depth_pivot.columns[1:])

    group_map = {
        'Early (0-3)':  'Early\n(0-3)',
        'Middle (4-7)': 'Middle\n(4-7)',
        'Late (8-11)':  'Late\n(8-11)',
        'early':        'Early\n(0-3)',
        'middle':       'Middle\n(4-7)',
        'late':         'Late\n(8-11)',
    }
    depth_pivot['group'] = depth_pivot['group'].map(lambda g: group_map.get(g, g))
    print(depth_pivot)
else:
    depth_pivot = pd.DataFrame({
        'group': ['Early\n(0-3)', 'Middle\n(4-7)', 'Late\n(8-11)'],
        'r256':  [0.854, 0.825, 0.333],
        'r128':  [0.625, 0.465, 0.000],
        'r64':   [0.370, 0.253, 0.000],
    })

In [ ]:
def plot_depth_sens(ax, df=depth_pivot):
    groups = df['group'].values
    x = np.arange(len(groups))
    rank_cols = [c for c in df.columns if c.startswith('r') and c != 'group']
    # Order descending rank (largest first)
    rank_cols = sorted(rank_cols, key=lambda c: int(c.replace('r', '')), reverse=True)
    width = 0.7 / len(rank_cols)
    rank_colors = [sty.SAGE, sty.TERRA_L, sty.TERRA, sty.SAND][:len(rank_cols)]

    for i, (col, rc) in enumerate(zip(rank_cols, rank_colors)):
        vals = df[col].values
        label = f'$r{{=}}{col.replace("r", "")}$'
        ax.bar(x + i * width, vals, width, label=label, color=rc,
               edgecolor=sty.BG, linewidth=0.8)
        for j, v in enumerate(vals):
            text = f'{v:.0%}' if v > 0.01 else '0%'
            color = sty.INK_3 if v > 0.01 else sty.TERRA
            y_pos = v + 0.02 if v > 0.01 else 0.02
            ax.text(x[j] + i * width, y_pos, text, ha='center',
                    fontsize=sty.SMALL_SIZE, color=color)

    ax.set_xticks(x + width * (len(rank_cols) - 1) / 2)
    ax.set_xticklabels(groups)
    ax.set_ylabel(sty.L['f1_retention'])
    ax.set_title(sty.L['t_depth_sens'])
    ax.set_ylim(0, 1.05)
    sty.format_pct(ax)
    ax.legend(fontsize=sty.LEGEND_SIZE)


sty.generate_both(plot_depth_sens, 'depth_sensitivity', figsize=sty.FIG_WIDE)

## Figura 7 — Frontera de Pareto (estrategias)

NB02 `spectral_analysis_results.csv` columns: `strategy, type, macro_f1,
retention, compression_ratio, f1_retention_pct`. The `type` field is already
set by NB02 — no inference needed.

In [ ]:
pareto_df = load_csv(CSV_MASTER_STRATEGY, name='NB02 strategy master table')

if pareto_df is not None:
    # Ensure f1_retention alias exists for plotting code consistency
    if 'f1_retention' not in pareto_df.columns and 'retention' in pareto_df.columns:
        pareto_df['f1_retention'] = pareto_df['retention']
    print(pareto_df[['strategy', 'type', 'compression_ratio', 'f1_retention']])
else:
    pareto_df = pd.DataFrame({
        'strategy': ['uniform_r512', 'uniform_r384', 'uniform_r256', 'uniform_r128',
                     'adaptive_e99', 'adaptive_e95', 'adaptive_e90', 'adaptive_e80'],
        'type':     ['uniform']*4 + ['adaptive']*4,
        'compression_ratio': [1.0, 0.806, 0.612, 0.418, 1.202, 1.022, 0.892, 0.721],
        'f1_retention':      [0.805, 0.435, 0.043, 0.0, 0.967, 0.850, 0.619, 0.248],
    })

In [ ]:
def plot_pareto(ax, df=pareto_df):
    for stype in [t for t in df['type'].unique() if t != 'baseline']:
        subset = df[df['type'] == stype].sort_values('compression_ratio')
        color = sty.strategy_color(stype)
        marker = sty.strategy_marker(stype)
        label = sty.L.get(stype, stype.capitalize())

        ax.plot(subset['compression_ratio'], subset['f1_retention'],
                color=color, lw=1.0, alpha=0.4, zorder=3)
        ax.scatter(subset['compression_ratio'], subset['f1_retention'],
                   color=color, marker=marker, s=70, edgecolor='white',
                   linewidth=0.8, zorder=5, label=label)

    ax.axhline(y=1.0, color=sty.INK_3, ls=':', lw=0.6, alpha=0.4)
    ax.axvline(x=1.0, color=sty.INK_3, ls=':', lw=0.6, alpha=0.4)
    ax.text(1.02, 0.02, sty.L['baseline_params'], fontsize=sty.SMALL_SIZE,
            color=sty.INK_3, style='italic')

    xmax = df['compression_ratio'].max() + 0.05
    if xmax > 1.05:
        ax.axvspan(1.0, xmax, alpha=0.03, color=sty.TERRA)
        ax.text((1.0 + xmax) / 2, 0.92, sty.L['expansion_zone'],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, ha='center', style='italic')

    ax.legend(loc='lower right')
    ax.set_xlabel(sty.L['param_ratio'])
    ax.set_ylabel(sty.L['f1_retention'])
    ax.set_title(sty.L['t_pareto'])
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)


sty.generate_both(plot_pareto, 'pareto_frontier', figsize=sty.FIG_FULL)

## Figura 8 — Distribución de rangos por umbral de energía

NB02 `adaptive_rank_distribution.csv` columns: `threshold (int, e.g. 99/95/90/80),
component, mean_rank, min_rank, max_rank`.

In [ ]:
arank_df = load_csv(CSV_ADAPTIVE_RANKS, name='adaptive ranks')

if arank_df is not None:
    def plot_adaptive_ranks(ax, df=arank_df):
        thresholds = sorted(df['threshold'].unique())
        components = df['component'].unique()

        x = np.arange(len(thresholds))
        n = len(components)
        w = 0.8 / n

        for i, comp in enumerate(components):
            sub = df[df['component'] == comp].sort_values('threshold')
            color = sty.component_color(comp)
            ax.bar(x + i*w - (n-1)*w/2, sub['mean_rank'].values, w,
                   label=comp.replace('_', ' '), color=color,
                   edgecolor=sty.BG, linewidth=0.5)

        ax.axhline(y=384, color=sty.INK_3, ls='--', lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 390, sty.L['break_even_attn'],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va='bottom')
        ax.axhline(y=614, color=sty.INK_3, ls='--', lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 620, sty.L['break_even_ffn'],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va='bottom')

        ax.set_xticks(x)
        ax.set_xticklabels([f'$\\tau$={t}%' for t in thresholds])
        ax.set_ylabel(sty.L['rank_assigned'])
        ax.set_title(sty.L['t_adaptive_ranks'])
        ax.set_ylim(0, 780)
        ax.legend(fontsize=sty.SMALL_SIZE, ncol=3, loc='upper left')
        ax.grid(axis='x', visible=False)

    sty.generate_both(plot_adaptive_ranks, 'adaptive_rank_distribution',
                      figsize=sty.FIG_WIDE)
else:
    print('-> Run NB02 export to produce adaptive_rank_distribution.csv')

## Figura 9 — Solapamiento de subespacios QKV

NB02 `qkv_subspace_overlap.csv` columns: `layer, pair, overlap_top_{rank}` for
several ranks. Shows the mean overlap between QK, QV, KV top-r subspaces
across layers.

In [ ]:
qkv_df = load_csv(CSV_QKV_OVERLAP, name='QKV subspace overlap')

if qkv_df is not None:
    def plot_qkv_overlap(ax, df=qkv_df):
        overlap_cols = sorted(
            [c for c in df.columns if c.startswith('overlap_top_')],
            key=lambda c: int(c.replace('overlap_top_', '')),
        )
        pairs = list(df['pair'].unique())

        mean_by_pair = {}
        for pair in pairs:
            sub = df[df['pair'] == pair]
            mean_by_pair[pair] = [sub[c].mean() for c in overlap_cols]

        x = [int(c.replace('overlap_top_', '')) for c in overlap_cols]
        pair_colors = [sty.BLUE, sty.TERRA, sty.SAGE, sty.SAND, sty.PLUM]
        for (pair, vals), color in zip(mean_by_pair.items(), pair_colors):
            ax.plot(x, vals, color=color, marker='o', markersize=6, lw=1.8,
                    markeredgecolor='white', markeredgewidth=0.8, label=pair)

        ax.set_xlabel('$r$ (top singular subspace)')
        ax.set_ylabel('Mean subspace overlap')
        ax.set_title('QKV subspace overlap (mean across layers)')
        ax.set_xlim(min(x) - 5, max(x) + 5)
        ax.set_ylim(0, 1.05)
        sty.format_pct(ax, decimals=0)
        ax.legend(loc='lower right')

    sty.generate_both(plot_qkv_overlap, 'qkv_overlap', figsize=sty.FIG_WIDE)
else:
    print('-> Run NB02 export to produce qkv_subspace_overlap.csv')

## Resumen

Todas las figuras se escriben en `figures/cap4_*_{es,en}.png` (300 DPI).
Para regenerar, re-ejecuta todas las celdas. Si algún CSV no existe, la celda
correspondiente se salta con un mensaje (salvo la Figura 3, que tiene fallback
incorporado).